<a href="https://colab.research.google.com/github/F1ameX/Modern-Methods-of-Deep-Machine-Learning/blob/main/4_CNN/4_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 33.9 MB/s eta 0:00:00


In [2]:
import os
import math
import random
from dataclasses import dataclass
from typing import Dict, Any, Tuple, List, Optional

import optuna
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

In [3]:
class ConvBlockA(nn.Module):
    def __init__(
        self,
        input_channels: int,
        output_channels: int,
        conv_kernel_size: int,
        conv_stride: int,
        pool_kernel_size: int,
        pool_stride: int,
    ):
        super().__init__()

        self.conv = nn.Conv2d(
            in_channels=input_channels,
            out_channels=output_channels,
            kernel_size=conv_kernel_size,
            stride=conv_stride,
            padding=conv_kernel_size // 2,
        )
        self.activation = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=pool_kernel_size, stride=pool_stride)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        x = self.activation(x)
        x = self.pool(x)
        return x


class ConvBlockB(nn.Module):
    def __init__(
        self,
        input_channels: int,
        output_channels: int,
        conv_kernel_size: int,
        conv_stride: int,
        pool_kernel_size: int,
        pool_stride: int,
    ):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=input_channels,
            out_channels=output_channels,
            kernel_size=conv_kernel_size,
            stride=conv_stride,
            padding=conv_kernel_size // 2,
        )
        self.conv2 = nn.Conv2d(
            in_channels=output_channels,
            out_channels=output_channels,
            kernel_size=conv_kernel_size,
            stride=1,
            padding=conv_kernel_size // 2,
        )

        self.activation = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=pool_kernel_size, stride=pool_stride)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.activation(x)

        x = self.conv2(x)
        x = self.activation(x)

        x = self.pool(x)
        return x


class CNN(nn.Module):
    def __init__(
        self,
        block_type: str = "a",
        n_blocks: int = 2,
        base_channels: int = 16,
        conv_kernel_size: int = 3,
        conv_stride: int = 1,
        pool_kernel_size: int = 2,
        pool_stride: int = 2,
        num_classes: int = 10,
    ):
        super().__init__()
        assert block_type in ("a", "b")
        assert n_blocks >= 1

        BlockClass = ConvBlockA if block_type == "a" else ConvBlockB

        blocks = []

        input_channels = 1
        channels = base_channels

        for _ in range(n_blocks):
            blocks.append(
                BlockClass(
                    input_channels=input_channels,
                    output_channels=channels,
                    conv_kernel_size=conv_kernel_size,
                    conv_stride=conv_stride,
                    pool_kernel_size=pool_kernel_size,
                    pool_stride=pool_stride,
                )
            )
            input_channels = channels
            channels *= 2

        self.features = nn.Sequential(*blocks)

        with torch.no_grad():
            dummy_x = torch.zeros(1, 1, 28, 28)
            dummy_out = self.features(dummy_x)
            flatten_dim = dummy_out.view(1, -1).shape[1]

        self.classifier = nn.Linear(flatten_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = x.flatten(1)
        logits = self.classifier(x)
        return logits